# Mathematical Modeling of Arterial Blood Pressure
## Windkessel Model Using Ordinary Differential Equations

**Author:** Zorana Savanović  
**University of Novi Sad**, Faculty of Sciences, Department of Mathematics and Informatics  
**Advisor:** Prof. Dr. Jelena Aleksić

---

This notebook presents a mathematical modeling approach to arterial blood pressure dynamics using the Windkessel model. We derive ordinary differential equations (ODEs) from the electrical circuit analogy of the cardiovascular system, solve them analytically and numerically, and validate the results through simulation.

**Key topics covered:**
- Electrical circuit analogy for blood flow (heart → current source, arterial compliance → capacitor, peripheral resistance → resistor)
- 2-element Windkessel model: derivation, analytical solution, and simulation
- 3-element Windkessel model: added proximal resistance for realistic systolic pressure
- Numerical methods: Euler and Runge-Kutta 4th order (RK4)
- Validation against analytical solutions
- Parameter sensitivity analysis (aging, hypertension)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Plot styling
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
    'figure.dpi': 100
})

# Color palette
BLUE = '#2E75B6'
ORANGE = '#E67E22'
RED = '#E74C3C'
GREEN = '#27AE60'
GRAY = '#888888'

## 1. Background: Cardiovascular Physiology

The heart pumps blood in discrete beats through the arterial system. During **systole**, the ventricles contract and eject blood into the aorta; during **diastole**, the ventricles relax and refill. This creates a pulsatile pressure waveform — typically peaking around 120 mmHg (systolic) and dropping to ~80 mmHg (diastolic) in a healthy adult at rest.

A key physiological feature is that blood flow remains relatively continuous despite the pulsatile pumping — this is due to the **Windkessel effect**: elastic arteries stretch during systole (storing blood) and recoil during diastole (releasing it), smoothing out pressure oscillations.

The Windkessel model, first proposed by Otto Frank in 1899, captures this effect using an analogy between hemodynamics and electrical circuits.

## 2. Model Parameters and Units

| Parameter | Symbol | Physiological Meaning | Unit | Typical Value |
|-----------|--------|----------------------|------|---------------|
| Peripheral resistance | $R$ (or $R_2$) | Resistance of small arteries and capillaries | mmHg·s/mL | 0.8–1.2 |
| Characteristic impedance | $R_1$ | Resistance of aorta and large arteries (3-element only) | mmHg·s/mL | 0.03–0.1 |
| Arterial compliance | $C$ | Ability of arteries to store blood volume | mL/mmHg | 1.0–2.0 |
| Cardiac output (flow) | $Q_{in}(t)$ | Blood flow from left ventricle into aorta | mL/s | 0–500 (pulsatile) |
| Arterial pressure | $P(t)$ | Blood pressure in arteries | mmHg | 80–120 |

### Dimensional consistency check

The governing equation is $C \cdot \frac{dP}{dt} + \frac{P}{R} = Q_{in}(t)$. Checking units:

- $C \cdot \frac{dP}{dt}$: $\frac{\text{mL}}{\text{mmHg}} \cdot \frac{\text{mmHg}}{\text{s}} = \frac{\text{mL}}{\text{s}}$ ✓
- $\frac{P}{R}$: $\frac{\text{mmHg}}{\text{mmHg·s/mL}} = \frac{\text{mL}}{\text{s}}$ ✓
- $Q_{in}$: mL/s ✓

## 3. Two-Element Windkessel Model

### 3.1 Model Derivation (Step 1: Model Formation)

The simplest Windkessel model uses two components in parallel:
- **Capacitor** $C$ — arterial compliance (arteries stretch and store blood)
- **Resistor** $R$ — peripheral vascular resistance (resistance to blood flow through small vessels)

The heart acts as a **current source** $Q_{in}(t)$. By Kirchhoff's current law (conservation of flow):

$$Q_{in}(t) = Q_C(t) + Q_R(t) = C\frac{dP}{dt} + \frac{P}{R}$$

Rearranging:

$$\boxed{\frac{dP}{dt} = \frac{1}{C}\left(Q_{in}(t) - \frac{P}{R}\right)}$$

This is a **first-order linear ODE** for arterial pressure $P(t)$.

### 3.2 Analytical Solution (Step 2: Model Analysis)

We solve the **initial value problem** with $P(0) = P_0$.

**Homogeneous solution** (diastole, $Q_{in} = 0$):

$$C\frac{dP}{dt} + \frac{P}{R} = 0 \implies \frac{dP}{P} = -\frac{dt}{RC} \implies P_h(t) = P_0 \cdot e^{-t/(RC)}$$

The product $\tau = RC$ is the **time constant** — it determines how quickly pressure decays during diastole.

**Particular solution** (constant $Q_{in}$): set $dP/dt = 0$:

$$0 = \frac{1}{C}\left(Q_{in} - \frac{P_p}{R}\right) \implies P_p = Q_{in} \cdot R$$

**General solution** of the initial value problem:

$$\boxed{P(t) = Q_{in} \cdot R + (P_0 - Q_{in} \cdot R) \cdot e^{-t/(RC)}}$$

As $t \to \infty$, pressure approaches the steady state $P = Q_{in} \cdot R$.

In [ ]:
def windkessel_2elem_euler(t, Qin, R, C, P0):
    """Solve 2-element Windkessel model using Euler's method.
    
    Parameters
    ----------
    t : array — time points (s)
    Qin : array — input flow at each time point (mL/s)
    R : float — peripheral resistance (mmHg·s/mL)
    C : float — arterial compliance (mL/mmHg)
    P0 : float — initial pressure (mmHg)
    
    Returns
    -------
    P : array — arterial pressure at each time point (mmHg)
    """
    dt = t[1] - t[0]
    P = np.zeros_like(t)
    P[0] = P0
    
    for i in range(1, len(t)):
        dPdt = (1/C) * (Qin[i-1] - P[i-1]/R)
        P[i] = P[i-1] + dPdt * dt
    
    return P

In [ ]:
# --- Simulation parameters ---
C = 1.5e-3    # arterial compliance (L/mmHg = 1.5 mL/mmHg)
R = 1.0       # peripheral resistance (mmHg·s/mL)
f = 1.0       # heart rate (Hz = 60 bpm)
A = 80.0      # flow amplitude (mL/s)
Q0 = 20.0     # baseline flow (mL/s)
P0 = 80.0     # initial pressure (mmHg)
dt = 0.001    # time step (s)
tmax = 4.0    # simulation duration (s)

t = np.arange(0, tmax, dt)

# Pulsatile input flow: positive half of a sine wave
Qin = Q0 + A * np.maximum(0, np.sin(2 * np.pi * f * t))

# --- Run 2-element simulation ---
P_2elem = windkessel_2elem_euler(t, Qin, R, C, P0)

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t, P_2elem, color=BLUE, lw=2, label='Arterial pressure P(t)')
ax.plot(t, Qin / np.max(Qin) * np.max(P_2elem) * 0.4, ':', 
        color=GRAY, label=r'Input flow $Q_{in}$ (scaled)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('P (mmHg)')
ax.set_title('2-Element Windkessel Model (Euler Method)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/sim_2elem.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Systolic (max): {P_2elem[len(t)//2:].max():.1f} mmHg')
print(f'Diastolic (min): {P_2elem[len(t)//2:].min():.1f} mmHg')
print(f'Time constant τ = RC = {R*C*1000:.1f} ms')

## 4. Three-Element Windkessel Model

### 4.1 Model Derivation

The 2-element model captures the exponential pressure decay in diastole well, but fails to reproduce the sharp pressure rise at the onset of systole seen in clinical measurements. The **3-element model** adds a resistor $R_1$ (characteristic impedance of the aorta) **in series** with the parallel $RC$ branch.

The ODE for the pressure $P(t)$ at the RC junction remains the same:

$$C\frac{dP}{dt} + \frac{P}{R_2} = Q_{in}(t)$$

The key difference is that the **measured aortic pressure** includes the voltage drop across $R_1$:

$$\boxed{P_a(t) = P(t) + R_1 \cdot Q_{in}(t)}$$

where:
- $P(t)$ is the **distal pressure** (at the RC branch, downstream of the aorta)
- $P_a(t)$ is the **aortic pressure** (what a catheter in the aorta would measure)
- The term $R_1 \cdot Q_{in}$ is proportional to flow, so it creates sharp peaks during systole

In [ ]:
def windkessel_3elem_rk4(t, Qin, R1, R2, C, P0):
    """Solve 3-element Windkessel model using RK4.
    
    Parameters
    ----------
    t : array — time points (s)
    Qin : array — input flow (mL/s)
    R1 : float — characteristic impedance (mmHg·s/mL)
    R2 : float — peripheral resistance (mmHg·s/mL)
    C : float — arterial compliance (mL/mmHg)
    P0 : float — initial distal pressure (mmHg)
    
    Returns
    -------
    P : array — distal pressure at RC junction (mmHg)
    Pa : array — aortic pressure P + R1*Qin (mmHg)
    """
    dt = t[1] - t[0]
    
    def dPdt(P_val, Qin_val):
        return (Qin_val - P_val / R2) / C
    
    P = np.zeros_like(t)
    P[0] = P0
    
    for i in range(len(t) - 1):
        Pi, qi = P[i], Qin[i]
        k1 = dPdt(Pi, qi)
        k2 = dPdt(Pi + 0.5*dt*k1, qi)
        k3 = dPdt(Pi + 0.5*dt*k2, qi)
        k4 = dPdt(Pi + dt*k3, qi)
        P[i+1] = Pi + dt * (k1 + 2*k2 + 2*k3 + k4) / 6.0
    
    Pa = P + R1 * Qin
    return P, Pa

In [ ]:
# --- 3-element model parameters ---
R1 = 0.05     # characteristic impedance (mmHg·s/mL)
R2 = 1.0      # peripheral resistance (mmHg·s/mL)

P_distal, P_aortic = windkessel_3elem_rk4(t, Qin, R1, R2, C, P0)

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t, P_aortic, color=BLUE, lw=2, label=r'Aortic pressure $P_a(t)$')
ax.plot(t, P_distal, '--', color=ORANGE, lw=1.5, label='Distal pressure P(t)')
ax.plot(t, Qin / np.max(Qin) * np.max(P_aortic) * 0.4, ':', 
        color=GRAY, label=r'$Q_{in}$ (scaled)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('P (mmHg)')
ax.set_title('3-Element Windkessel Model (RK4 Method)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/sim_3elem.png', dpi=150, bbox_inches='tight')
plt.show()

half = len(t) // 2
print(f'Aortic — Systolic: {P_aortic[half:].max():.1f} mmHg, '
      f'Diastolic: {P_aortic[half:].min():.1f} mmHg')
print(f'Distal — Systolic: {P_distal[half:].max():.1f} mmHg, '
      f'Diastolic: {P_distal[half:].min():.1f} mmHg')

## 5. Model Validation

### 5.1 Constant Flow: Numerical vs. Analytical Solution

To verify the numerical implementation, we compare the Euler solution against the known analytical result for constant input flow $Q_{in} = 100$ mL/s:

$$P(t) = Q_{in} \cdot R + (P_0 - Q_{in} \cdot R) \cdot e^{-t/(RC)}$$

In [ ]:
# --- Validation: constant Qin ---
Qin_const_val = 100.0  # mL/s
Qin_const_arr = np.full_like(t, Qin_const_val)

# Analytical solution
P_steady = Qin_const_val * R
tau = R * C
P_analytical = P_steady + (P0 - P_steady) * np.exp(-t / tau)

# Numerical solution (Euler)
P_numerical = windkessel_2elem_euler(t, Qin_const_arr, R, C, P0)

# Error
error = np.abs(P_analytical - P_numerical)

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), height_ratios=[3, 1])

ax1.plot(t, P_analytical, color=BLUE, lw=2, label='Analytical solution')
ax1.plot(t[::50], P_numerical[::50], 'o', color=RED, markersize=4, 
         label=f'Euler (dt={dt})')
ax1.axhline(y=P_steady, color=GREEN, ls='--', alpha=0.5, 
            label=f'Steady state P = {P_steady:.0f} mmHg')
ax1.set_ylabel('P (mmHg)')
ax1.set_title(f'Validation: Constant $Q_{{in}}$ = {Qin_const_val:.0f} mL/s, '
              f'P(0) = {P0:.0f} mmHg')
ax1.legend()

ax2.plot(t, error, color=RED, lw=1.5)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('|Error| (mmHg)')
ax2.set_title('Absolute error: Euler vs. Analytical')

plt.tight_layout()
plt.savefig('../figures/validation_constant_qin.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Max error: {error.max():.6f} mmHg')
print(f'Time constant τ = {tau*1000:.3f} ms')

### 5.2 Comparison: 2-Element vs. 3-Element Model

Setting $R_1 = 0$ in the 3-element model reduces it to the 2-element model. The comparison below highlights how the additional series resistance $R_1$ creates sharper systolic peaks that better match physiological measurements.

In [ ]:
# --- 2-elem via RK4 (R1=0 makes it equivalent) ---
_, P_2elem_rk4 = windkessel_3elem_rk4(t, Qin, R1=0, R2=R, C=C, P0=P0)
_, P_3elem_rk4 = windkessel_3elem_rk4(t, Qin, R1=R1, R2=R2, C=C, P0=P0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t, P_2elem_rk4, color=ORANGE, lw=2, label='2-element: P(t)')
ax.plot(t, P_3elem_rk4, color=BLUE, lw=2, label=r'3-element: $P_a(t)$')
ax.plot(t, Qin / np.max(Qin) * np.max(P_3elem_rk4) * 0.35, ':', 
        color=GRAY, alpha=0.7, label=r'$Q_{in}$ (scaled)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('P (mmHg)')
ax.set_title('2-Element vs. 3-Element Windkessel Model')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/comparison_2v3.png', dpi=150, bbox_inches='tight')
plt.show()

half = len(t) // 2
print(f'2-elem pulse pressure: '
      f'{P_2elem_rk4[half:].max() - P_2elem_rk4[half:].min():.1f} mmHg')
print(f'3-elem pulse pressure: '
      f'{P_3elem_rk4[half:].max() - P_3elem_rk4[half:].min():.1f} mmHg')

## 6. Parameter Sensitivity Analysis

Understanding how model parameters affect pressure dynamics has direct clinical relevance:

- **Decreased compliance** $C$ (aging, atherosclerosis) → larger pulse pressure, stiffer arteries
- **Increased resistance** $R$ (hypertension) → elevated mean arterial pressure

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# --- Vary compliance C ---
C_values = [
    (0.5e-3, 'C = 0.5 mL/mmHg (reduced — aging)', RED),
    (1.5e-3, 'C = 1.5 mL/mmHg (normal)', BLUE),
    (3.0e-3, 'C = 3.0 mL/mmHg (increased)', GREEN)
]

for C_val, label, color in C_values:
    _, Pa = windkessel_3elem_rk4(t, Qin, R1, R2, C_val, P0)
    ax1.plot(t, Pa, color=color, lw=1.8, label=label)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel(r'$P_a$ (mmHg)')
ax1.set_title('Effect of Arterial Compliance (C)')
ax1.legend(fontsize=9)

# --- Vary resistance R ---
R_values = [
    (0.5, 'R = 0.5 (low resistance)', GREEN),
    (1.0, 'R = 1.0 (normal)', BLUE),
    (2.0, 'R = 2.0 (hypertension)', RED)
]

for R_val, label, color in R_values:
    _, Pa = windkessel_3elem_rk4(t, Qin, R1, R_val, C, P0)
    ax2.plot(t, Pa, color=color, lw=1.8, label=label)

ax2.set_xlabel('Time (s)')
ax2.set_ylabel(r'$P_a$ (mmHg)')
ax2.set_title('Effect of Peripheral Resistance (R)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Quantitative sensitivity summary ---
print('=== Sensitivity Analysis: Steady-State Metrics ===')
print(f'{"Config":<40} {"Systolic":>10} {"Diastolic":>10} {"Pulse":>10} {"Mean":>10}')
print('-' * 82)

half = len(t) // 2

configs = [
    ('C=0.5 (reduced compliance)', R1, R2, 0.5e-3),
    ('C=1.5 (normal)', R1, R2, 1.5e-3),
    ('C=3.0 (high compliance)', R1, R2, 3.0e-3),
    ('R=0.5 (low resistance)', R1, 0.5, C),
    ('R=1.0 (normal)', R1, 1.0, C),
    ('R=2.0 (hypertension)', R1, 2.0, C),
]

for name, r1, r2, c in configs:
    _, Pa = windkessel_3elem_rk4(t, Qin, r1, r2, c, P0)
    sys_p = Pa[half:].max()
    dia_p = Pa[half:].min()
    pulse = sys_p - dia_p
    mean_p = np.mean(Pa[half:])
    print(f'{name:<40} {sys_p:>9.1f} {dia_p:>10.1f} {pulse:>10.1f} {mean_p:>10.1f}')

## 7. Conclusions

1. **The 2-element Windkessel model** produces a first-order linear ODE whose analytical solution (exponential decay + particular solution) matches numerical simulations with negligible error.

2. **The 3-element model** adds a series resistance $R_1$ that creates the clinically observed sharp systolic pressure rise, without changing the underlying ODE for the RC branch.

3. **Parameter sensitivity** connects directly to clinical phenomena:
   - Reduced $C$ (arterial stiffening) → increased pulse pressure
   - Increased $R$ (vasoconstriction) → elevated mean arterial pressure (hypertension)

4. Despite its simplicity, the Windkessel model provides valuable insight into cardiovascular dynamics and remains widely used in biomedical engineering research and education.

### Limitations and extensions

- **No wave propagation:** Windkessel models are lumped-parameter models — they cannot capture pulse wave reflections or propagation delays along the arterial tree.
- **4-element models** add an inductor (blood inertia) to improve high-frequency response.
- **Distributed models** (1D transmission line) provide full wave dynamics but require significantly more parameters and computational effort.

## References

1. Boyce, W. E., & DiPrima, R. C. *Elementary Differential Equations and Boundary Value Problems.* 9th ed., Wiley, 2009.
2. Frank, O. "Die Grundform des Arteriellen Pulses." *Zeitschrift für Biologie*, 37, 483–526, 1899.
3. Westerhof, N., Lankhaar, J.W., Westerhof, B.E. "The arterial Windkessel." *Med Biol Eng Comput*, 47, 131–141, 2009.
4. Hlavac, M. "Windkessel model analysis." [ecmosimulation.com](https://www.ecmosimulation.com/data/Hlavac.pdf)